<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Modelado de Entradas
### T3.1 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Proceso metodológico de modelado de entradas
2. **Ejemplo 1:** Ajuste de distribución exponencial a datos de servicio
3. Exploración gráfica: histograma, ECDF, Q-Q
4. **Ejemplo 2:** Banco — análisis completo de llegadas y servicios
5. Pruebas de bondad de ajuste y selección de modelos (AIC)
6. **Ejemplo 3:** Taller industrial — Weibull, Log-normal y comparación
7. Impacto de la distribución en el modelo de colas

## Metodología de modelado de entradas — 5 pasos
<div class='defn'>

1. **Recolección de datos**: verificar independencia y homogeneidad
2. **Exploración gráfica**: histograma, ECDF, Q-Q plot
3. **Identificación de la familia**: CV, soporte, proceso físico
4. **Estimación de parámetros**: MLE o método de momentos
5. **Prueba de bondad de ajuste**: χ², KS, Anderson-Darling
</div>

**Guía rápida por CV:**
- CV = 0 → Determinista
- 0 < CV < 1 → Erlang-k o Gamma
- CV ≈ 1 → Exponencial
- CV > 1 → Hiperexponencial, Weibull (k<1), Log-normal

## Ejemplo 1 — Ventanilla bancaria (n=15)
<div class='defn'>
Un banco registra 15 tiempos de servicio (minutos). ¿Siguen una distribución exponencial?
</div>

**Datos (ordenados):** 0.7, 0.8, 0.9, 1.1, 1.2, 1.5, 1.6, 1.8, 2.1, 2.4, 2.8, 3.1, 3.4, 4.2, 5.6

**Estadísticos muestrales:**
- x̄ = 2.21 min, s = 1.32 min
- **CV = 0.60** → más regular que la exponencial (CV=1)

<div class='nota'>
<strong>Señal clave:</strong> CV < 1 sugiere Erlang o Gamma, pero con n=15 la incertidumbre del CV es alta.
</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Datos del banco
data_bank = np.array([0.7, 0.8, 0.9, 1.1, 1.2, 1.5, 1.6, 1.8, 2.1, 2.4, 2.8, 3.1, 3.4, 4.2, 5.6])
n = len(data_bank)
xbar = data_bank.mean()
sd = data_bank.std(ddof=1)
cv = sd / xbar
print(f'n = {n},  x̄ = {xbar:.2f} min,  s = {sd:.2f} min,  CV = {cv:.2f}')

# MLE exponencial
lam_hat = 1 / xbar
print(f'\nMLE Exponencial: λ̂ = 1/x̄ = {lam_hat:.3f} min⁻¹')

# Gráficos: histograma + Q-Q exponencial
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histograma
axes[0].hist(data_bank, bins=5, density=True, color='#1A7A9A', edgecolor='white', alpha=0.8)
x_pdf = np.linspace(0, 6, 200)
axes[0].plot(x_pdf, stats.expon.pdf(x_pdf, scale=xbar), 'r-', lw=2, label=f'Exp(λ={lam_hat:.2f})')
axes[0].set_xlabel('Tiempo (min)'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Histograma + PDF exponencial'); axes[0].legend()

# ECDF vs CDF teórica
data_sorted = np.sort(data_bank)
ecdf_y = np.arange(1, n+1) / n
axes[1].step(data_sorted, ecdf_y, where='post', color='#1A7A9A', lw=2, label='ECDF')
axes[1].plot(x_pdf, stats.expon.cdf(x_pdf, scale=xbar), 'r--', lw=2, label='CDF Exp')
axes[1].set_xlabel('Tiempo (min)'); axes[1].set_ylabel('F(x)')
axes[1].set_title('ECDF vs CDF exponencial'); axes[1].legend()

# Q-Q plot
stats.probplot(data_bank, dist='expon', sparams=(0, xbar), plot=axes[2])
axes[2].set_title('Q-Q plot exponencial')
axes[2].get_lines()[0].set_color('#1A7A9A')
axes[2].get_lines()[1].set_color('#C62828')

plt.tight_layout(); plt.show()

# Prueba chi-cuadrado (3 celdas equiprobables)
a1 = stats.expon.ppf(1/3, scale=xbar)
a2 = stats.expon.ppf(2/3, scale=xbar)
obs = [np.sum(data_bank < a1), np.sum((data_bank >= a1) & (data_bank < a2)), np.sum(data_bank >= a2)]
exp_counts = [n/3]*3
chi2_stat = sum((o-e)**2/e for o, e in zip(obs, exp_counts))
print(f'\nPrueba χ² (k=3 celdas): χ² = {chi2_stat:.2f},  χ²(0.05, 1) = 3.84')
print(f'Resultado: {"No se rechaza" if chi2_stat < 3.84 else "Se rechaza"} H₀ (exponencial)')
print(f'\n⚠️  Con n=15, la prueba tiene muy bajo poder. El CV=0.60 es la señal más informativa.')

## Herramientas gráficas de diagnóstico

| Herramienta | Qué muestra | Limitación |
|---|---|---|
| **Histograma** | Forma, asimetría, modas | Sensible a k (# bins) |
| **ECDF** | Ajuste global sin bins | No resalta colas |
| **Q-Q plot** | Desviaciones en colas | Requiere elegir distribución |

**Patrones del Q-Q plot:**
- Puntos sobre la diagonal en cola derecha → datos con colas más pesadas
- Curvatura en S → asimetría diferente al modelo
- Pendiente ≠ 1 → escala mal estimada

<div class='nota'>
Siempre usar <strong>al menos dos</strong> herramientas gráficas. El histograma es la primera impresión; el Q-Q es el diagnóstico fino.
</div>

## Ejemplo 2 — Banco completo (n=80)
<div class='defn'>
Se amplía el estudio del banco: 80 tiempos entre llegadas y 80 tiempos de servicio.
Objetivo: determinar si el sistema es M/M/1, M/G/1 o G/G/1.
</div>

**Plan:**
1. Analizar llegadas → ¿Poisson? (CV ≈ 1)
2. Analizar servicios → ¿Exponencial? ¿Erlang? ¿Gamma?
3. Comparar candidatas con AIC
4. Cuantificar impacto en W_q

In [ ]:
np.random.seed(2026)

# Simular datos del banco (Erlang-3 para servicio, Exp para llegadas)
arrival_times = stats.expon.rvs(scale=3.2, size=80)
service_times = stats.erlang.rvs(a=3, scale=2.18/3, size=80)

# ─── Análisis de llegadas ───
print('═══ TIEMPOS ENTRE LLEGADAS ═══')
print(f'n = {len(arrival_times)},  x̄ = {arrival_times.mean():.2f} min,  s = {arrival_times.std(ddof=1):.2f} min')
print(f'CV = {arrival_times.std(ddof=1)/arrival_times.mean():.2f}')

# KS test para exponencialidad
lam_a = 1/arrival_times.mean()
ks_a, p_a = stats.kstest(arrival_times, 'expon', args=(0, arrival_times.mean()))
print(f'KS test: D = {ks_a:.4f}, p-value = {p_a:.4f} → {"No se rechaza" if p_a > 0.05 else "Se rechaza"} Exp')

# ─── Análisis de servicios ───
print(f'\n═══ TIEMPOS DE SERVICIO ═══')
print(f'n = {len(service_times)},  x̄ = {service_times.mean():.2f} min,  s = {service_times.std(ddof=1):.2f} min')
print(f'CV = {service_times.std(ddof=1)/service_times.mean():.2f}')
print(f'k estimado ≈ 1/CV² = {1/(service_times.std(ddof=1)/service_times.mean())**2:.1f} → Erlang-3 o Gamma')

# Ajustar candidatas
candidatas = {}

# Exponencial
scale_exp = service_times.mean()
ll_exp = np.sum(stats.expon.logpdf(service_times, scale=scale_exp))
candidatas['Exponencial'] = {'params': f'λ={1/scale_exp:.3f}', 'LL': ll_exp, 'k': 1,
                              'AIC': -2*ll_exp + 2*1, 'dist': stats.expon(scale=scale_exp)}

# Erlang-3 (gamma con a=3)
scale_e3 = service_times.mean()/3
ll_e3 = np.sum(stats.erlang.logpdf(service_times, a=3, scale=scale_e3))
candidatas['Erlang-3'] = {'params': f'μf={1/scale_e3:.3f}', 'LL': ll_e3, 'k': 1,
                          'AIC': -2*ll_e3 + 2*1, 'dist': stats.erlang(a=3, scale=scale_e3)}

# Gamma (forma y escala libres)
alpha_g, _, beta_g = stats.gamma.fit(service_times, floc=0)
ll_g = np.sum(stats.gamma.logpdf(service_times, alpha_g, scale=beta_g))
candidatas['Gamma'] = {'params': f'α={alpha_g:.2f}, β={1/beta_g:.2f}', 'LL': ll_g, 'k': 2,
                       'AIC': -2*ll_g + 2*2, 'dist': stats.gamma(alpha_g, scale=beta_g)}

print(f'\n{"Distribución":<14} {"Parámetros":<22} {"Log-L":>8} {"AIC":>8}')
print('-'*56)
for name, info in candidatas.items():
    ks, pv = stats.kstest(service_times, info['dist'].cdf)
    print(f'{name:<14} {info["params"]:<22} {info["LL"]:>8.1f} {info["AIC"]:>8.1f}   KS p={pv:.3f}')

best = min(candidatas, key=lambda x: candidatas[x]['AIC'])
print(f'\n✓ Mejor modelo por AIC: {best}')

In [ ]:
# Gráficos comparativos de las candidatas
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x_s = np.linspace(0, service_times.max()*1.1, 300)
colors = {'Exponencial': '#C62828', 'Erlang-3': '#1A7A9A', 'Gamma': '#C8961E'}

# Histograma + PDFs
axes[0].hist(service_times, bins=12, density=True, color='#BBDEFB', edgecolor='white', alpha=0.8, label='Datos')
for name, info in candidatas.items():
    axes[0].plot(x_s, info['dist'].pdf(x_s), color=colors[name], lw=2, label=f'{name} (AIC={info["AIC"]:.0f})')
axes[0].set_xlabel('Tiempo de servicio (min)'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Histograma + PDFs candidatas'); axes[0].legend(fontsize=8)

# ECDF vs CDFs
data_s = np.sort(service_times)
ecdf_s = np.arange(1, len(data_s)+1) / len(data_s)
axes[1].step(data_s, ecdf_s, where='post', color='black', lw=2, label='ECDF')
for name, info in candidatas.items():
    axes[1].plot(x_s, info['dist'].cdf(x_s), color=colors[name], lw=2, ls='--', label=name)
axes[1].set_xlabel('Tiempo (min)'); axes[1].set_ylabel('F(x)')
axes[1].set_title('ECDF vs CDFs candidatas'); axes[1].legend(fontsize=8)

# Q-Q Erlang-3 (la ganadora)
theor_q = candidatas['Erlang-3']['dist'].ppf((np.arange(1, len(data_s)+1) - 0.5) / len(data_s))
axes[2].scatter(theor_q, data_s, color='#1A7A9A', s=20, zorder=3)
lims = [0, max(data_s.max(), theor_q.max())*1.05]
axes[2].plot(lims, lims, 'r--', lw=1.5, label='y = x')
axes[2].set_xlabel('Cuantiles teóricos (Erlang-3)'); axes[2].set_ylabel('Cuantiles empíricos')
axes[2].set_title('Q-Q plot: Erlang-3'); axes[2].legend()

plt.tight_layout(); plt.show()

# Impacto en Wq
lam_bank = 1/arrival_times.mean()
ES = service_times.mean()
rho = lam_bank * ES
ES2 = service_times.var() + ES**2
Wq_MM1 = rho / ((1/ES)*(1-rho))
Wq_MG1 = lam_bank * ES2 / (2*(1-rho))
print(f'\n═══ IMPACTO EN EL MODELO DE COLAS ═══')
print(f'ρ = {rho:.3f}')
print(f'Si usamos M/M/1 (exponencial): Wq = {Wq_MM1:.2f} min')
print(f'Si usamos M/E₃/1 (Erlang-3):   Wq = {Wq_MG1:.2f} min  (P-K)')
print(f'Error de usar M/M/1: +{(Wq_MM1/Wq_MG1-1)*100:.0f}%')

## Pruebas de bondad de ajuste

| Prueba | Estadístico | Ventaja | Limitación |
|---|---|---|---|
| **χ²** | Σ(O-E)²/E | Versátil, discreta y continua | Requiere E≥5 por celda |
| **KS** | sup|F̂ₙ-F₀| | Sin bins, continua | Anticonservativa con θ̂ estimado |
| **AD** | Ponderada en colas | Más potente en colas | Valores críticos por distribución |

## Selección de modelos: AIC y BIC
<div class='defn'>

$$\text{AIC} = -2\ell(\hat\theta) + 2p, \qquad \text{BIC} = -2\ell(\hat\theta) + p\ln n$$

Menor AIC/BIC = mejor modelo. Si ΔAIC < 2, los modelos son indistinguibles.
</div>

## Ejemplo 3 — Taller de reparación industrial
<div class='defn'>
Un taller atiende equipos industriales. Se registran 50 tiempos entre llegadas y 50 tiempos de reparación. Las reparaciones son heterogéneas (diagnósticos rápidos vs. revisiones completas).
</div>

**Diferencia clave con el banco:**
- El banco tenía servicio en **etapas regulares** → Erlang (CV < 1)
- El taller tiene reparaciones **heterogéneas** → ¿Weibull? ¿Log-normal?
- La tasa de falla puede ser no constante (desgaste durante la reparación)

In [ ]:
np.random.seed(31)

# Datos del taller (Weibull para reparación, Exp para llegadas)
arrival_taller = stats.expon.rvs(scale=4.8, size=50)
repair_times = stats.weibull_min.rvs(c=1.28, scale=3.97, size=50)

print('═══ TIEMPOS ENTRE LLEGADAS ═══')
print(f'x̄ = {arrival_taller.mean():.2f} h,  CV = {arrival_taller.std(ddof=1)/arrival_taller.mean():.2f}')
ks_arr, p_arr = stats.kstest(arrival_taller, 'expon', args=(0, arrival_taller.mean()))
print(f'KS test exponencial: p = {p_arr:.3f} → {"No se rechaza" if p_arr > 0.05 else "Se rechaza"}')

print(f'\n═══ TIEMPOS DE REPARACIÓN ═══')
print(f'x̄ = {repair_times.mean():.2f} h,  s = {repair_times.std(ddof=1):.2f} h')
print(f'CV = {repair_times.std(ddof=1)/repair_times.mean():.2f}')

# Ajustar 4 candidatas
cands = {}

# Exponencial
sc_e = repair_times.mean()
ll_e = stats.expon.logpdf(repair_times, scale=sc_e).sum()
cands['Exponencial'] = {'AIC': -2*ll_e+2, 'dist': stats.expon(scale=sc_e), 'p': 1}

# Gamma
ag, _, bg = stats.gamma.fit(repair_times, floc=0)
ll_gm = stats.gamma.logpdf(repair_times, ag, scale=bg).sum()
cands['Gamma'] = {'AIC': -2*ll_gm+4, 'dist': stats.gamma(ag, scale=bg), 'p': 2}

# Weibull
cw, _, sw = stats.weibull_min.fit(repair_times, floc=0)
ll_w = stats.weibull_min.logpdf(repair_times, cw, scale=sw).sum()
cands['Weibull'] = {'AIC': -2*ll_w+4, 'dist': stats.weibull_min(cw, scale=sw), 'p': 2}

# Log-normal
sig_ln, _, sc_ln = stats.lognorm.fit(repair_times, floc=0)
ll_ln = stats.lognorm.logpdf(repair_times, sig_ln, scale=sc_ln).sum()
cands['Log-normal'] = {'AIC': -2*ll_ln+4, 'dist': stats.lognorm(sig_ln, scale=sc_ln), 'p': 2}

print(f'\n{"Distribución":<14} {"AIC":>8}  {"KS D":>7}  {"KS p":>7}  {"AD A²":>7}')
print('-'*52)
for name, info in cands.items():
    ks_d, ks_p = stats.kstest(repair_times, info['dist'].cdf)
    ad_stat = stats.anderson_ksamp([repair_times, info['dist'].rvs(size=500, random_state=42)])
    print(f'{name:<14} {info["AIC"]:>8.1f}  {ks_d:>7.4f}  {ks_p:>7.3f}')

best3 = min(cands, key=lambda x: cands[x]['AIC'])
print(f'\n✓ Mejor modelo por AIC: {best3}')
if best3 == 'Weibull':
    print(f'  k = {cw:.2f} > 1 → tasa de falla CRECIENTE (desgaste durante reparación)')

In [ ]:
# Gráficos del taller
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x_r = np.linspace(0.01, repair_times.max()*1.2, 300)
cols3 = {'Exponencial':'#C62828', 'Gamma':'#2E7D32', 'Weibull':'#1A7A9A', 'Log-normal':'#C8961E'}

# Histograma + PDFs
axes[0].hist(repair_times, bins=10, density=True, color='#BBDEFB', edgecolor='white', alpha=0.8)
for name, info in cands.items():
    axes[0].plot(x_r, info['dist'].pdf(x_r), color=cols3[name], lw=2,
                 label=f'{name} (AIC={info["AIC"]:.0f})')
axes[0].set_xlabel('Tiempo de reparación (h)'); axes[0].set_ylabel('Densidad')
axes[0].set_title('Reparaciones del taller'); axes[0].legend(fontsize=7)

# Q-Q Weibull
data_r = np.sort(repair_times)
theor_w = cands['Weibull']['dist'].ppf((np.arange(1, len(data_r)+1) - 0.5) / len(data_r))
axes[1].scatter(theor_w, data_r, color='#1A7A9A', s=25, zorder=3)
lims2 = [0, max(data_r.max(), theor_w.max())*1.05]
axes[1].plot(lims2, lims2, 'r--', lw=1.5, label='y = x')
axes[1].set_xlabel('Cuantiles Weibull'); axes[1].set_ylabel('Cuantiles empíricos')
axes[1].set_title('Q-Q plot: Weibull'); axes[1].legend()

# Tasa de falla (hazard) para diferentes distribuciones
t_haz = np.linspace(0.1, 10, 200)
axes[2].plot(t_haz, stats.expon.pdf(t_haz, scale=sc_e)/stats.expon.sf(t_haz, scale=sc_e),
             color='#C62828', lw=2, label='Exp (constante)')
axes[2].plot(t_haz, stats.weibull_min.pdf(t_haz, cw, scale=sw)/stats.weibull_min.sf(t_haz, cw, scale=sw),
             color='#1A7A9A', lw=2, label=f'Weibull k={cw:.2f} (creciente)')
axes[2].set_xlabel('Tiempo (h)'); axes[2].set_ylabel('h(t)')
axes[2].set_title('Tasa de falla (hazard)'); axes[2].legend()

plt.tight_layout(); plt.show()

# Impacto en Wq
lam_t = 1/arrival_taller.mean()
ES_t = repair_times.mean()
rho_t = lam_t * ES_t
ES2_t = repair_times.var() + ES_t**2
Wq_MM1_t = rho_t / ((1/ES_t)*(1-rho_t))
Wq_MG1_t = lam_t * ES2_t / (2*(1-rho_t))
print(f'\n═══ IMPACTO EN MODELO DE COLAS ═══')
print(f'ρ = {rho_t:.3f}')
print(f'M/M/1 (exponencial): Wq = {Wq_MM1_t:.1f} h')
print(f'M/G/1 (Weibull):     Wq = {Wq_MG1_t:.1f} h  (P-K)')
print(f'Diferencia: {(Wq_MM1_t/Wq_MG1_t-1)*100:+.0f}%')

## ¿Qué hacer sin datos?

| Estrategia | Cuándo usar | Distribución |
|---|---|---|
| **Elicitación de expertos** | Mínimo, moda y máximo conocidos | Triangular, PERT |
| **Valores de literatura** | Benchmarks industriales | Depende |
| **Análisis de sensibilidad** | Entrada incierta | Varias |
| **Distribución uniforme** | Solo se conocen límites | U(a,b) |

<div class='alerta'>
<strong>⚠️ La triangular no es un sustituto permanente.</strong> Sus colas lineales no representan tiempos de servicio/falla realistas. Siempre buscar datos reales para el modelo definitivo.
</div>

## Conclusiones

- El **CV** es el estadístico más informativo para seleccionar la familia de distribución.
- Con **muestras pequeñas** (n<20), las pruebas formales tienen bajo poder; confiar en gráficos y CV.
- Con **muestras grandes** (n>200), las pruebas rechazan desviaciones irrelevantes; usar AIC/BIC.
- Siempre verificar **independencia** (autocorrelación lag-1, runs test) y **estacionariedad** antes de ajustar.
- La elección de distribución impacta directamente en Wq: puede subestimar o sobreestimar por un **factor de 2×**.
- Si ninguna distribución ajusta: verificar datos, considerar mezclas, o usar la ECDF directamente.

**Próximo tema:** T3.2 — Verificación y Validación del modelo de simulación.